In [1]:
# Import libraries and define configs
import json
import random
import warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import resample

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.figsize": (10, 6),
        "figure.dpi": 120,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
    }
)

RANDOM_STATE = 121
N_JOBS = -1
CV_FOLDS = 5
OPERATING_THRESHOLD = 0.065424
OUTREACH_CAPACITIES = (0.01, 0.05, 0.10, 0.20)

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

current_directory = Path.cwd().resolve()
project_root_candidates = [current_directory, *current_directory.parents]

PROJECT_ROOT = next(
    (
        candidate_directory
        for candidate_directory in project_root_candidates
        if (candidate_directory / "data" / "processed").exists()
        and (candidate_directory / "notebooks").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing data/processed and notebooks."
    )

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modeling"
PRIVATE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "private"
INTERPRETABILITY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "interpretability"

INTERPRETABILITY_TABLES_DIR = INTERPRETABILITY_OUTPUT_DIR / "tables"
INTERPRETABILITY_FIGURES_DIR = INTERPRETABILITY_OUTPUT_DIR / "figures"

INTERPRETABILITY_TABLES_DIR.mkdir(parents=True, exist_ok=True)
INTERPRETABILITY_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DONOR_FEATURES_PARQUET_PATH = PROCESSED_DATA_DIR / "donor_features.parquet"
DONOR_FEATURES_CSV_PATH = PROCESSED_DATA_DIR / "donor_features.csv"
FEATURE_DICTIONARY_PATH = REPORTS_DIR / "feature_dictionary.csv"

PRIMARY_PIPELINE_PATH = MODELS_DIR / "final_primary_pipeline.joblib"
BENCHMARK_PIPELINE_PATH = (
    MODELS_DIR / "historical_donor_status_benchmark_pipeline.joblib"
)

FINAL_MODEL_CONFIGURATION_PATH = (
    MODELING_OUTPUT_DIR / "final_model_configuration.json"
)
FINAL_FEATURE_LISTS_PATH = MODELING_OUTPUT_DIR / "final_feature_lists.json"
FINAL_TEST_PREDICTIONS_PATH = (
    PRIVATE_OUTPUT_DIR / "primary_donor_predictions_final_test.csv"
)
PRIMARY_TEST_METRICS_PATH = (
    MODELING_OUTPUT_DIR / "primary_final_test_metrics.csv"
)
PRIMARY_OUTREACH_RESULTS_PATH = (
    MODELING_OUTPUT_DIR / "primary_final_test_outreach_results.csv"
)
BENCHMARK_TEST_METRICS_PATH = (
    MODELING_OUTPUT_DIR / "benchmark_test_metrics.csv"
)
MODELING_ARTIFACT_MANIFEST_PATH = (
    MODELING_OUTPUT_DIR / "modeling_artifact_manifest.csv"
)

In [2]:
print(f"Project root: {PROJECT_ROOT.name}")
print(f"Random state: {RANDOM_STATE}")
print(f"Operating threshold: {OPERATING_THRESHOLD:.6f}")
print("Phase 6 interpretability setup complete.")

Project root: red-cross-donor-prediction
Random state: 121
Operating threshold: 0.065424
Phase 6 interpretability setup complete.


In [3]:
# Load models, metadata, predicts, and eval artifacts
BENCHMARK_MODEL_COMPARISON_PATH = (
    MODELING_OUTPUT_DIR / "benchmark_model_comparison.csv"
)

required_phase5_artifact_paths = {
    "Primary model pipeline": PRIMARY_PIPELINE_PATH,
    "Benchmark model pipeline": BENCHMARK_PIPELINE_PATH,
    "Final model configuration": FINAL_MODEL_CONFIGURATION_PATH,
    "Final feature lists": FINAL_FEATURE_LISTS_PATH,
    "Feature dictionary": FEATURE_DICTIONARY_PATH,
    "Primary test predictions": FINAL_TEST_PREDICTIONS_PATH,
    "Primary test metrics": PRIMARY_TEST_METRICS_PATH,
    "Primary outreach results": PRIMARY_OUTREACH_RESULTS_PATH,
    "Benchmark model comparison": BENCHMARK_MODEL_COMPARISON_PATH,
    "Benchmark test metrics": BENCHMARK_TEST_METRICS_PATH,
    "Modeling artifact manifest": MODELING_ARTIFACT_MANIFEST_PATH,
}

missing_phase5_artifacts = {
    artifact_name: artifact_path
    for artifact_name, artifact_path in required_phase5_artifact_paths.items()
    if not artifact_path.exists()
}

if missing_phase5_artifacts:
    missing_artifact_list = "\n".join(
        f"- {artifact_name}: {artifact_path.relative_to(PROJECT_ROOT)}"
        for artifact_name, artifact_path in missing_phase5_artifacts.items()
    )
    raise FileNotFoundError(
        f"Required Phase 5 artifacts were not found:\n{missing_artifact_list}"
    )

model_primary_pipeline_final = joblib.load(PRIMARY_PIPELINE_PATH)
model_benchmark_pipeline_final = joblib.load(BENCHMARK_PIPELINE_PATH)

with FINAL_MODEL_CONFIGURATION_PATH.open("r", encoding="utf-8") as file:
    configuration_final_model = json.load(file)

with FINAL_FEATURE_LISTS_PATH.open("r", encoding="utf-8") as file:
    feature_sets_final = json.load(file)

metadata_feature_dictionary = pd.read_csv(FEATURE_DICTIONARY_PATH)
predictions_primary_final_test = pd.read_csv(FINAL_TEST_PREDICTIONS_PATH)

tracking_primary_final_test_donor_ids = (
    predictions_primary_final_test.loc[:, ["donor_unique_id"]].copy()
)

metrics_primary_final_test = pd.read_csv(PRIMARY_TEST_METRICS_PATH)
metrics_primary_outreach = pd.read_csv(PRIMARY_OUTREACH_RESULTS_PATH)
comparison_benchmark_models = pd.read_csv(BENCHMARK_MODEL_COMPARISON_PATH)
metrics_benchmark_final_test = pd.read_csv(BENCHMARK_TEST_METRICS_PATH)
manifest_modeling_artifacts = pd.read_csv(MODELING_ARTIFACT_MANIFEST_PATH)

primary_pipeline_steps = getattr(
    model_primary_pipeline_final,
    "named_steps",
    {},
)
benchmark_pipeline_steps = getattr(
    model_benchmark_pipeline_final,
    "named_steps",
    {},
)

preprocessor_primary_model_final = primary_pipeline_steps.get("preprocessor")
preprocessor_benchmark_model_final = benchmark_pipeline_steps.get("preprocessor")

if preprocessor_primary_model_final is None:
    primary_preprocessor_candidates = [
        pipeline_step
        for pipeline_step in primary_pipeline_steps.values()
        if hasattr(pipeline_step, "transformers_")
    ]

    if len(primary_preprocessor_candidates) == 1:
        preprocessor_primary_model_final = primary_preprocessor_candidates[0]
    else:
        raise KeyError(
            "The fitted primary preprocessing step could not be identified."
        )

if preprocessor_benchmark_model_final is None:
    benchmark_preprocessor_candidates = [
        pipeline_step
        for pipeline_step in benchmark_pipeline_steps.values()
        if hasattr(pipeline_step, "transformers_")
    ]

    if len(benchmark_preprocessor_candidates) == 1:
        preprocessor_benchmark_model_final = benchmark_preprocessor_candidates[0]
    else:
        raise KeyError(
            "The fitted benchmark preprocessing step could not be identified."
        )

loaded_artifact_summary = pd.DataFrame(
    [
        {
            "Artifact": "Primary model pipeline",
            "Object Type": type(model_primary_pipeline_final).__name__,
            "Rows": pd.NA,
            "Columns": pd.NA,
            "Source": PRIMARY_PIPELINE_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        {
            "Artifact": "Benchmark model pipeline",
            "Object Type": type(model_benchmark_pipeline_final).__name__,
            "Rows": pd.NA,
            "Columns": pd.NA,
            "Source": BENCHMARK_PIPELINE_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        {
            "Artifact": "Feature dictionary",
            "Object Type": type(metadata_feature_dictionary).__name__,
            "Rows": metadata_feature_dictionary.shape[0],
            "Columns": metadata_feature_dictionary.shape[1],
            "Source": FEATURE_DICTIONARY_PATH.relative_to(PROJECT_ROOT).as_posix(),
        },
        {
            "Artifact": "Primary final-test predictions",
            "Object Type": type(predictions_primary_final_test).__name__,
            "Rows": predictions_primary_final_test.shape[0],
            "Columns": predictions_primary_final_test.shape[1],
            "Source": FINAL_TEST_PREDICTIONS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Primary final-test metrics",
            "Object Type": type(metrics_primary_final_test).__name__,
            "Rows": metrics_primary_final_test.shape[0],
            "Columns": metrics_primary_final_test.shape[1],
            "Source": PRIMARY_TEST_METRICS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Primary outreach results",
            "Object Type": type(metrics_primary_outreach).__name__,
            "Rows": metrics_primary_outreach.shape[0],
            "Columns": metrics_primary_outreach.shape[1],
            "Source": PRIMARY_OUTREACH_RESULTS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Benchmark model comparison",
            "Object Type": type(comparison_benchmark_models).__name__,
            "Rows": comparison_benchmark_models.shape[0],
            "Columns": comparison_benchmark_models.shape[1],
            "Source": BENCHMARK_MODEL_COMPARISON_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Benchmark final-test metrics",
            "Object Type": type(metrics_benchmark_final_test).__name__,
            "Rows": metrics_benchmark_final_test.shape[0],
            "Columns": metrics_benchmark_final_test.shape[1],
            "Source": BENCHMARK_TEST_METRICS_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
        {
            "Artifact": "Modeling artifact manifest",
            "Object Type": type(manifest_modeling_artifacts).__name__,
            "Rows": manifest_modeling_artifacts.shape[0],
            "Columns": manifest_modeling_artifacts.shape[1],
            "Source": MODELING_ARTIFACT_MANIFEST_PATH.relative_to(
                PROJECT_ROOT
            ).as_posix(),
        },
    ]
)

display(
    loaded_artifact_summary.style
    .hide(axis="index")
    .set_properties(**{"text-align": "center", "padding": "8px"})
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("padding", "8px"),
                ],
            }
        ]
    )
    .format(
        {
            "Rows": "{:,.0f}",
            "Columns": "{:,.0f}",
        },
        na_rep="—",
    )
)

print(f"Primary pipeline steps: {list(primary_pipeline_steps)}")
print(f"Benchmark pipeline steps: {list(benchmark_pipeline_steps)}")
print(
    "Final-test donor IDs loaded: "
    f"{tracking_primary_final_test_donor_ids.shape[0]:,}"
)
print("Phase 5 artifacts loaded successfully.")

Artifact,Object Type,Rows,Columns,Source
Primary model pipeline,Pipeline,—,—,models/final_primary_pipeline.joblib
Benchmark model pipeline,Pipeline,—,—,models/historical_donor_status_benchmark_pipeline.joblib
Feature dictionary,DataFrame,77,7,reports/feature_dictionary.csv
Primary final-test predictions,DataFrame,"6,881",7,outputs/private/primary_donor_predictions_final_test.csv
Primary final-test metrics,DataFrame,10,2,outputs/modeling/primary_final_test_metrics.csv
Primary outreach results,DataFrame,4,8,outputs/modeling/primary_final_test_outreach_results.csv
Benchmark model comparison,DataFrame,5,20,outputs/modeling/benchmark_model_comparison.csv
Benchmark final-test metrics,DataFrame,10,2,outputs/modeling/benchmark_test_metrics.csv
Modeling artifact manifest,DataFrame,12,5,outputs/modeling/modeling_artifact_manifest.csv


Primary pipeline steps: ['preprocessor', 'classifier']
Benchmark pipeline steps: ['preprocessor', 'classifier']
Final-test donor IDs loaded: 6,881
Phase 5 artifacts loaded successfully.
